# Probar modelo

Cargue un modelo `.h5` entrenado y pruébelo con imágenes, dataset completo, o cámara en vivo.

| Paso | Descripción |
|------|-------------|
| 1 | Setup |
| 2 | Cargar modelo .h5 (auto-detecta clases y config desde .json) |
| 3 | Probar con imágenes |
| 4 | Comparar modelos (tabla con todos los .json en modelos/) |
| 5 | Evaluar dataset completo |
| 6 | Cámara en vivo (OpenCV) |

> No entrena modelos — para eso use `entrenar.ipynb`.
> Al exportar en `entrenar.ipynb` se genera un `.json` con metadata que este notebook lee automáticamente.

---
## 1. Setup

In [ ]:
import os, glob, random, io
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator, load_img, img_to_array
)
from sklearn.metrics import classification_report, confusion_matrix
import ipywidgets as widgets
from IPython.display import display, clear_output

random.seed(42)
np.random.seed(42)

# Estado
cfg = {
    'model': None, 'class_names': [], 'img_size': 128,
    'preprocessing': 'rescale', 'dataset_dir': '',
}

def _detectar_preprocesamiento(model):
    """Detecta si el modelo usa MobileNetV2."""
    for layer in model.layers:
        if 'mobilenet' in layer.name.lower():
            return 'mobilenet'
    return 'rescale'

def _preprocess(img_array, preprocessing):
    """Preprocesa imagen según tipo de modelo."""
    if preprocessing == 'mobilenet':
        from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
        return preprocess_input(img_array.astype('float32'))
    return img_array.astype('float32') / 255.0

print(f"TensorFlow {tf.__version__} — Listo")

---
## 2. Cargar modelo .h5

Solo necesita configurar `MODEL_PATH`. Si el modelo tiene un `.json` asociado
(generado por `entrenar.ipynb`), se auto-detectan las clases, preprocesamiento y dataset.

Si no hay `.json` (modelo viejo), puede configurar `CLASS_NAMES` manualmente.

In [ ]:
# --- Listar modelos disponibles ---
modelos_encontrados = []
for d in ['modelos', '.']:
    if os.path.isdir(d):
        modelos_encontrados += glob.glob(os.path.join(d, '*.h5'))
        modelos_encontrados += glob.glob(os.path.join(d, '*.keras'))

if modelos_encontrados:
    print("Modelos encontrados:")
    for m in sorted(modelos_encontrados):
        mb = os.path.getsize(m) / 1024 / 1024
        print(f"  {m} ({mb:.1f} MB)")
else:
    print("No se encontraron modelos en modelos/ ni en el directorio actual.")
    print("Primero entrene un modelo con entrenar.ipynb")

In [ ]:
# ============================================================
# CARGAR MODELO — seleccione del dropdown o configure manualmente
# ============================================================

import json as _json

# --- Descubrir modelos con metadata ---
_modelos_disponibles = {}
for d in ['modelos', '.']:
    if not os.path.isdir(d):
        continue
    for ext in ('*.h5', '*.keras'):
        for h5 in glob.glob(os.path.join(d, ext)):
            json_path = h5.rsplit('.', 1)[0] + '.json'
            meta = {}
            if os.path.exists(json_path):
                with open(json_path) as f:
                    meta = _json.load(f)
            mb = os.path.getsize(h5) / 1024 / 1024
            va = meta.get('val_accuracy')
            label = os.path.basename(h5)
            if va is not None:
                label += f"  (val {va:.0%})"
            label += f"  [{mb:.0f}MB]"
            _modelos_disponibles[label] = {'path': h5, 'meta': meta}

def _cargar_modelo(model_path, class_names_manual=None):
    """Carga modelo y actualiza cfg global."""
    if not os.path.exists(model_path):
        print(f"✘ No se encontró: {model_path}")
        return

    model = load_model(model_path, compile=False)
    model.compile(optimizer='adam', loss='categorical_crossentropy',
                  metrics=['accuracy'])

    img_size = model.input_shape[1]
    n_clases = model.output_shape[-1]
    preprocessing = _detectar_preprocesamiento(model)

    json_path = model_path.rsplit('.', 1)[0] + '.json'
    metadata = {}
    if os.path.exists(json_path):
        with open(json_path) as f:
            metadata = _json.load(f)
        print(f"✔ Metadata cargada: {os.path.basename(json_path)}")

    # class_names: JSON > manual > dataset > genérico
    class_names = []
    if metadata.get('class_names'):
        class_names = metadata['class_names']
        print(f"  Clases (del JSON): {class_names}")
    elif class_names_manual:
        class_names = class_names_manual
        print(f"  Clases (manual): {class_names}")
    else:
        dataset_dir = metadata.get('dataset_dir', '../data/mi_dataset')
        if os.path.isdir(dataset_dir):
            class_names = sorted([d for d in os.listdir(dataset_dir)
                                  if os.path.isdir(os.path.join(dataset_dir, d))])
            if len(class_names) != n_clases:
                class_names = [f'Clase {i}' for i in range(n_clases)]
        if not class_names:
            class_names = [f'Clase {i}' for i in range(n_clases)]
            print(f"  ⚠ Complete CLASS_NAMES manualmente.")

    if metadata.get('preprocessing'):
        preprocessing = metadata['preprocessing']

    dataset_dir = metadata.get('dataset_dir', '../data/mi_dataset')

    cfg['model'] = model
    cfg['img_size'] = img_size
    cfg['preprocessing'] = preprocessing
    cfg['class_names'] = class_names
    cfg['dataset_dir'] = dataset_dir
    cfg['model_path'] = model_path
    cfg['metadata'] = metadata

    tipo = 'MobileNetV2' if preprocessing == 'mobilenet' else 'CNN custom'
    print(f"\n✔ Modelo cargado: {os.path.basename(model_path)}")
    print(f"  Tipo: {tipo}")
    print(f"  Input: {img_size}×{img_size} | Clases: {n_clases}")
    print(f"  Clases: {class_names}")
    print(f"  Params: {model.count_params():,}")
    if metadata.get('val_accuracy'):
        print(f"  Val accuracy (entrenamiento): {metadata['val_accuracy']:.1%}")
    if metadata.get('dataset_dir'):
        print(f"  Dataset: {metadata['dataset_dir']}")

# --- Widget dropdown ---
if _modelos_disponibles:
    _w_dropdown = widgets.Dropdown(
        options=list(_modelos_disponibles.keys()),
        description='Modelo:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='100%'),
    )
    _w_btn_cargar = widgets.Button(
        description=' Cargar modelo',
        button_style='success', icon='check',
        layout=widgets.Layout(width='180px'),
    )
    _w_manual_names = widgets.Text(
        value='', placeholder="ej: rojo, azul, verde (solo si no hay .json)",
        description='Clases manual:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='100%'),
    )
    _w_out_load = widgets.Output()

    def _on_cargar(_):
        with _w_out_load:
            clear_output(wait=True)
            sel = _w_dropdown.value
            info = _modelos_disponibles[sel]
            manual = None
            if _w_manual_names.value.strip():
                manual = [c.strip() for c in _w_manual_names.value.split(',')]
            _cargar_modelo(info['path'], manual)

    _w_btn_cargar.on_click(_on_cargar)

    display(widgets.VBox([
        _w_dropdown,
        _w_manual_names,
        _w_btn_cargar,
        _w_out_load
    ]))
    print("Seleccione un modelo y presione 'Cargar modelo'.")
else:
    print("No se encontraron modelos en modelos/ ni en el directorio actual.")
    print("Primero entrene un modelo con entrenar.ipynb")
    print("\nO configure manualmente:")
    print("  MODEL_PATH = 'modelos/COMPLETAR.h5'")
    print("  _cargar_modelo(MODEL_PATH)")

---
## 3. Probar con imágenes

Pruebe el modelo con imágenes individuales: suba una foto o tome una aleatoria del dataset.

In [ ]:
def predecir_y_mostrar(img_pil, titulo=''):
    """Predice y muestra barras de confianza."""
    model = cfg['model']
    names = cfg['class_names']
    sz = cfg['img_size']

    img_resized = img_pil.resize((sz, sz))
    arr = np.array(img_resized)
    arr_proc = _preprocess(arr, cfg['preprocessing'])
    preds = model.predict(np.expand_dims(arr_proc, 0), verbose=0)[0]

    idx = np.argmax(preds)
    conf = preds[idx]

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4),
                                  gridspec_kw={'width_ratios': [1, 1.2]})
    a1.imshow(img_resized); a1.axis('off')
    c = 'green' if conf > 0.7 else 'orange' if conf > 0.4 else 'red'
    a1.set_title(f'{names[idx]} ({conf:.1%})', fontsize=14,
                 color=c, fontweight='bold')

    colors = ['#2ecc71' if i == idx else '#bdc3c7' for i in range(len(names))]
    bars = a2.barh(names, preds, color=colors)
    a2.set_xlim(0, 1); a2.set_xlabel('Confianza')
    for bar, val in zip(bars, preds):
        a2.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                f'{val:.1%}', va='center', fontsize=10)
    if titulo: fig.suptitle(titulo, fontsize=11)
    plt.tight_layout(); plt.show()
    return names[idx], conf

# --- Botones ---
w_up = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False,
                          description='Subir foto')
w_btn_pred = widgets.Button(description=' Predecir foto',
                            button_style='warning', icon='eye')
w_btn_rand = widgets.Button(description=' Imagen aleatoria del dataset',
                            button_style='info', icon='random')
w_out_test = widgets.Output()

def _pred_upload(_):
    with w_out_test:
        clear_output(wait=True)
        if not cfg['model']: print("\u2718 Cargue un modelo primero."); return
        if not w_up.value: print("\u2718 Suba una imagen."); return
        info = w_up.value
        content = (list(info.values())[0]['content']
                   if isinstance(info, dict) else info[0].content)
        from PIL import Image
        pil = Image.open(io.BytesIO(content)).convert('RGB')
        pred, conf = predecir_y_mostrar(pil, titulo='Imagen subida')
        print(f"Predicción: {pred} ({conf:.1%})")

def _rand_img(_):
    with w_out_test:
        clear_output(wait=True)
        if not cfg['model']: print("\u2718 Cargue un modelo primero."); return
        ddir = cfg['dataset_dir']
        if not os.path.isdir(ddir):
            print(f"\u2718 No se encontró: {ddir}"); return
        imgs = []
        for ext in ('*.jpg', '*.jpeg', '*.png'):
            imgs += glob.glob(os.path.join(ddir, '**', ext), recursive=True)
        if not imgs: print("No hay imágenes."); return
        p = random.choice(imgs)
        real = os.path.basename(os.path.dirname(p))
        from PIL import Image
        pil = Image.open(p).convert('RGB')
        pred, conf = predecir_y_mostrar(pil, titulo=f'Real: {real}')
        ok = '\u2714' if pred == real else '\u2718'
        print(f"{ok} Real: {real} | Predicción: {pred} ({conf:.1%})")

w_btn_pred.on_click(_pred_upload)
w_btn_rand.on_click(_rand_img)

display(widgets.VBox([
    widgets.HBox([w_up, w_btn_pred]),
    w_btn_rand,
    w_out_test
]))

---
## 4. Comparar modelos

Tabla comparativa de todos los modelos en `modelos/` que tienen metadata `.json`.
Útil para elegir el mejor modelo después de varios experimentos.

In [ ]:
# ============================================================
# COMPARAR MODELOS — tabla de todos los .json en modelos/
# ============================================================

import json as _json

json_files = sorted(glob.glob('modelos/*.json'))

if not json_files:
    print("No se encontraron archivos .json en modelos/")
    print("Entrene y exporte modelos desde entrenar.ipynb para generar metadata.")
else:
    modelos_info = []
    for jf in json_files:
        with open(jf) as f:
            meta = _json.load(f)
        h5_path = jf.rsplit('.', 1)[0] + '.h5'
        meta['_archivo'] = os.path.basename(h5_path)
        meta['_json_path'] = jf
        meta['_exists'] = os.path.exists(h5_path)
        modelos_info.append(meta)

    # Ordenar por val_accuracy descendente
    modelos_info.sort(key=lambda m: m.get('val_accuracy', 0), reverse=True)

    # Modelo actualmente cargado
    loaded = os.path.basename(cfg.get('model_path', '')) if cfg.get('model') else ''

    # Tabla
    print(f"{'':>3} {'Modelo':<42} {'Tipo':<10} {'Res':>4} {'Val':>7} {'Train':>7} {'Ep':>4} {'Fecha':<16}")
    print(f"{'':>3} {'─'*42} {'─'*10} {'─'*4} {'─'*7} {'─'*7} {'─'*4} {'─'*16}")

    for i, m in enumerate(modelos_info):
        marker = ' >>' if m['_archivo'] == loaded else f'{i+1:3d}'
        va = f"{m.get('val_accuracy', 0):.1%}" if m.get('val_accuracy') else '  —'
        ta = f"{m.get('train_accuracy', 0):.1%}" if m.get('train_accuracy') else '  —'
        ep = f"{m.get('epochs', '—'):>4}" if m.get('epochs') else '   —'
        res = f"{m.get('img_size', '?')}px"
        tipo = m.get('model_type', '?')
        fecha = m.get('timestamp', '—')
        exists = '' if m['_exists'] else ' [.h5 no encontrado]'
        print(f"{marker} {m['_archivo']:<42} {tipo:<10} {res:>4} {va:>7} {ta:>7} {ep} {fecha}{exists}")

    if loaded:
        print(f"\n >> = modelo actualmente cargado")

    print(f"\nTotal: {len(modelos_info)} modelo(s) con metadata")

---
## 5. Evaluar dataset completo

Confusion matrix, classification report, y galería de errores.
El dataset se obtiene automáticamente de la metadata del modelo.

In [ ]:
model = cfg['model']
names = cfg['class_names']
sz = cfg['img_size']
ddir = cfg['dataset_dir']

if model is None:
    print("\u2718 Cargue un modelo primero (celda 2).")
elif not os.path.isdir(ddir):
    print(f"\u2718 No se encontró dataset: {ddir}")
    print("  Verifique que el dataset existe o que el .json tiene la ruta correcta.")
else:
    # Generador con el preprocesamiento correcto
    if cfg['preprocessing'] == 'mobilenet':
        from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
        gen = ImageDataGenerator(preprocessing_function=preprocess_input)
    else:
        gen = ImageDataGenerator(rescale=1./255)

    flow = gen.flow_from_directory(
        ddir, target_size=(sz, sz), batch_size=32,
        class_mode='categorical', shuffle=False, classes=names
    )

    print(f"Evaluando {flow.samples} imágenes de {ddir}...")
    loss, acc = model.evaluate(flow, verbose=0)
    flow.reset()
    preds = model.predict(flow, verbose=0)
    y_pred = np.argmax(preds, axis=1)
    y_true = flow.classes
    labels = list(flow.class_indices.keys())

    print(f"\nAccuracy: {acc:.1%}\n")
    print(classification_report(y_true, y_pred,
                                target_names=labels, zero_division=0))

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(max(5, len(labels)*1.5), max(4, len(labels)*1.2)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicción'); plt.ylabel('Real')
    plt.title(f'Accuracy: {acc:.1%}')
    plt.tight_layout(); plt.show()

    # Galería de errores
    errs = [(i, flow.filenames[i]) for i in range(len(y_true))
            if y_pred[i] != y_true[i]]
    if errs:
        n = min(8, len(errs))
        print(f"\nErrores: {len(errs)} de {len(y_true)}")
        cols = min(4, n)
        rows = (n + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
        if rows == 1 and cols == 1: axes = np.array([axes])
        for ax in np.array(axes).ravel(): ax.axis('off')
        for j, ax in enumerate(np.array(axes).ravel()):
            if j >= n: break
            idx, fname = errs[j]
            p = os.path.join(ddir, fname)
            if os.path.exists(p):
                ax.imshow(load_img(p, target_size=(sz, sz)))
                ax.set_title(f"Real: {labels[y_true[idx]]}\n"
                             f"Pred: {labels[y_pred[idx]]} ({preds[idx][y_pred[idx]]:.0%})",
                             fontsize=9, color='red')
        plt.suptitle('Errores', fontsize=14)
        plt.tight_layout(); plt.show()
    else:
        print("\n\u2714 Sin errores.")

---
## 6. Cámara en vivo

Clasificación en tiempo real con interfaz visual.

**Controles:**
- `q` — Salir
- `s` — Guardar captura
- `d` — Mostrar/ocultar panel de confianza

> En WSL2 no hay acceso a cámara. Use `scripts/prueba_video.py` desde Windows.

In [ ]:
import cv2, time
from collections import deque

# ============================================================
# CONFIGURACIÓN DE CÁMARA
# ============================================================
CAMERA_INDEX = 0          # cambiar a 1 si no detecta
CONFIDENCE_THRESHOLD = 0.6
SMOOTHING_WINDOW = 5      # promedio últimas N predicciones (anti-parpadeo)

# ============================================================

model = cfg['model']
names = cfg['class_names']
sz = cfg['img_size']
preprocessing = cfg['preprocessing']

# Colores por clase (BGR)
CLASE_COLORES = [
    (76, 175, 80),   # Verde
    (255, 152, 0),   # Naranja
    (33, 150, 243),  # Azul
    (0, 188, 212),   # Cyan
    (156, 39, 176),  # Púrpura
    (255, 235, 59),  # Amarillo
]

def draw_ui(frame, class_name, confidence, all_preds, fps, show_panel):
    """Dibuja interfaz sobre el frame de la cámara."""
    h, w = frame.shape[:2]
    pred_idx = np.argmax(all_preds)
    color = CLASE_COLORES[pred_idx % len(CLASE_COLORES)]

    # Barra superior (resultado principal)
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (w, 60), (30, 30, 30), -1)
    cv2.addWeighted(overlay, 0.85, frame, 0.15, 0, frame)

    if confidence >= CONFIDENCE_THRESHOLD:
        label = f"{class_name}  {confidence*100:.0f}%"
        label_color = color
        cv2.rectangle(frame, (0, 0), (6, 60), color, -1)
    else:
        label = f"?  {confidence*100:.0f}%"
        label_color = (128, 128, 128)
        cv2.rectangle(frame, (0, 0), (6, 60), (128, 128, 128), -1)

    cv2.putText(frame, label, (16, 42),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, label_color, 2, cv2.LINE_AA)

    fps_text = f"{fps:.0f} FPS"
    cv2.putText(frame, fps_text, (w - 90, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 180, 180), 1, cv2.LINE_AA)

    # Panel lateral de confianza
    if show_panel:
        panel_w = 220
        panel_h = 30 + len(names) * 36
        panel_x = w - panel_w - 10
        panel_y = 70

        overlay2 = frame.copy()
        cv2.rectangle(overlay2, (panel_x, panel_y),
                      (panel_x + panel_w, panel_y + panel_h),
                      (20, 20, 20), -1)
        cv2.addWeighted(overlay2, 0.8, frame, 0.2, 0, frame)

        cv2.putText(frame, "Confianza", (panel_x + 10, panel_y + 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (180, 180, 180), 1, cv2.LINE_AA)

        bar_w = panel_w - 20
        for i, (name, prob) in enumerate(zip(names, all_preds)):
            y = panel_y + 32 + i * 36

            cv2.putText(frame, name, (panel_x + 10, y + 2),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4,
                        (220, 220, 220) if i == pred_idx else (140, 140, 140),
                        1, cv2.LINE_AA)

            bar_y = y + 8
            cv2.rectangle(frame, (panel_x + 10, bar_y),
                          (panel_x + 10 + bar_w, bar_y + 14),
                          (60, 60, 60), -1)

            fill_w = max(2, int(bar_w * prob))
            bar_color = CLASE_COLORES[i % len(CLASE_COLORES)] if i == pred_idx else (100, 100, 100)
            cv2.rectangle(frame, (panel_x + 10, bar_y),
                          (panel_x + 10 + fill_w, bar_y + 14),
                          bar_color, -1)

            cv2.putText(frame, f"{prob*100:.0f}%",
                        (panel_x + 10 + bar_w + 2, bar_y + 12),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.35,
                        (180, 180, 180), 1, cv2.LINE_AA)

    # Barra inferior (controles)
    overlay3 = frame.copy()
    cv2.rectangle(overlay3, (0, h - 30), (w, h), (30, 30, 30), -1)
    cv2.addWeighted(overlay3, 0.7, frame, 0.3, 0, frame)
    cv2.putText(frame, "Q salir  |  S captura  |  D panel",
                (10, h - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.4,
                (140, 140, 140), 1, cv2.LINE_AA)

    return frame

if model is None:
    print("\u2718 Cargue un modelo primero (celda 2).")
else:
    # Detectar WSL
    is_wsl = False
    try:
        with open('/proc/version', 'r') as f:
            is_wsl = 'microsoft' in f.read().lower()
    except: pass

    if is_wsl:
        print("\u26a0 WSL2 no tiene acceso a cámara USB.")
        print("Ejecute desde PowerShell/CMD en Windows:")
        print(f"  python scripts/prueba_video.py")
        print(f"\nCon esta configuración:")
        print(f'  MODEL_PATH = "proyecto/{os.path.basename(cfg.get("model_path", "modelo.h5"))}"')
        print(f'  IMG_SIZE = {sz}')
        print(f'  CLASS_NAMES = {names}')
    else:
        cap = cv2.VideoCapture(CAMERA_INDEX)
        if not cap.isOpened():
            print(f"\u2718 No se pudo abrir cámara (índice {CAMERA_INDEX})")
        else:
            cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
            cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
            prev_time = 0
            show_panel = True
            capture_count = 0

            # --- Suavizado temporal y carpeta de capturas ---
            pred_buffer = deque(maxlen=SMOOTHING_WINDOW)
            model_name = os.path.splitext(os.path.basename(cfg.get('model_path', 'modelo')))[0]
            capture_dir = os.path.join('capturas', model_name)
            os.makedirs(capture_dir, exist_ok=True)

            print("Cámara abierta. Ventana de OpenCV activa.")
            print(f"  Suavizado: últimas {SMOOTHING_WINDOW} predicciones")
            print(f"  Capturas se guardan en: {capture_dir}/")

            try:
                while True:
                    ret, frame = cap.read()
                    if not ret: break

                    curr_time = time.time()
                    fps = 1 / (curr_time - prev_time) if prev_time > 0 else 0
                    prev_time = curr_time

                    img = cv2.resize(frame, (sz, sz))
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    img_proc = _preprocess(img, preprocessing)
                    preds = model.predict(np.expand_dims(img_proc, 0), verbose=0)[0]

                    # Suavizado temporal (media móvil)
                    pred_buffer.append(preds)
                    smooth_preds = np.mean(pred_buffer, axis=0)

                    pred_idx = np.argmax(smooth_preds)
                    confidence = smooth_preds[pred_idx]
                    class_name = names[pred_idx]

                    frame = draw_ui(frame, class_name, confidence,
                                    smooth_preds, fps, show_panel)

                    cv2.imshow('PUCP - Clasificacion en Tiempo Real', frame)

                    key = cv2.waitKey(1) & 0xFF
                    if key == ord('q'): break
                    elif key == ord('s'):
                        path = os.path.join(capture_dir, f"captura_{capture_count:03d}.jpg")
                        cv2.imwrite(path, frame)
                        print(f"  Captura: {path}")
                        capture_count += 1
                    elif key == ord('d'):
                        show_panel = not show_panel

            except KeyboardInterrupt:
                print("Interrumpido")
            finally:
                cap.release()
                cv2.destroyAllWindows()
                print("Cámara cerrada.")